# Garmin Login + End-to-End Test (All-in-Databricks)

This self-contained notebook does everything needed to test the Garmin integration
without any local Python setup:

1. Creates the Unity Catalog schema and bronze table (if they don't exist)
2. Logs in to Garmin Connect using your email/password
3. Saves OAuth tokens to Databricks Secrets (so future runs don't need the password)
4. Extracts yesterday's health data from Garmin Connect
5. Writes raw JSON to the `wearables_zerobus` bronze table
6. Runs verification queries

**You only need to provide your Garmin email and password once.** After the first run,
tokens are stored in Databricks Secrets and refreshed automatically.

In [ ]:
%pip install garminconnect>=0.2.0

## Configuration

Set the widget values before running. For the first run, you **must** provide
your Garmin email and password. On subsequent runs, leave them blank and the
notebook will use saved tokens from Databricks Secrets.

In [ ]:
DEFAULT_CATALOG = "sailesh"
DEFAULT_SCHEMA = "wearables"
DEFAULT_SECRET_SCOPE = "dbxw_zerobus_credentials"
DEFAULT_DEVICE_ID = "garmin_forerunner_265"

try:
    dbutils.widgets.text("catalog", DEFAULT_CATALOG, "1. Catalog")
    dbutils.widgets.text("schema", DEFAULT_SCHEMA, "2. Schema")
    dbutils.widgets.text("secret_scope", DEFAULT_SECRET_SCOPE, "3. Secret Scope")
    dbutils.widgets.text("garmin_email", "", "4. Garmin Email (first run only)")
    dbutils.widgets.text("garmin_password", "", "5. Garmin Password (first run only)")
    dbutils.widgets.text("target_date", "", "6. Target Date (YYYY-MM-DD, blank=yesterday)")
    dbutils.widgets.text("device_id", DEFAULT_DEVICE_ID, "7. Device ID")

    catalog = dbutils.widgets.get("catalog")
    schema = dbutils.widgets.get("schema")
    secret_scope = dbutils.widgets.get("secret_scope")
    garmin_email = dbutils.widgets.get("garmin_email")
    garmin_password = dbutils.widgets.get("garmin_password")
    target_date_str = dbutils.widgets.get("target_date")
    device_id = dbutils.widgets.get("device_id")
except Exception:
    catalog = DEFAULT_CATALOG
    schema = DEFAULT_SCHEMA
    secret_scope = DEFAULT_SECRET_SCOPE
    garmin_email = ""
    garmin_password = ""
    target_date_str = ""
    device_id = DEFAULT_DEVICE_ID

spark.sql(f"USE CATALOG {catalog}")

bronze_table = f"{catalog}.{schema}.wearables_zerobus"

print(f"Catalog:      {catalog}")
print(f"Schema:       {schema}")
print(f"Bronze table: {bronze_table}")
print(f"Secret scope: {secret_scope}")
print(f"Garmin email: {'(provided)' if garmin_email else '(not provided -- will use saved tokens)'}")
print(f"Device ID:    {device_id}")

## Step 1: Create Schema and Bronze Table

Idempotent -- safe to run repeatedly. Matches the DDL from `zeroBus/dbxW_zerobus_infra`.

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
print(f"Schema {catalog}.{schema} ready")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {bronze_table}
    (
      record_id STRING NOT NULL COMMENT 'Server-generated GUID for each ingested record',
      ingested_at TIMESTAMP COMMENT 'Server-side ingestion timestamp',
      body VARIANT COMMENT 'Raw JSON payload stored as VARIANT',
      headers VARIANT COMMENT 'HTTP request headers as JSON',
      record_type STRING COMMENT 'Extracted from X-Record-Type header',
      CONSTRAINT wearables_zerobus_pk PRIMARY KEY (record_id)
    )
    USING DELTA
    COMMENT 'Bronze table for wearable health data'
    TBLPROPERTIES (
      'delta.enableChangeDataFeed' = 'true',
      'quality' = 'bronze'
    )
""")
print(f"Table {bronze_table} ready")

## Step 2: Create Secret Scope (if needed)

In [ ]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

existing_scopes = [s.name for s in w.secrets.list_scopes()]
if secret_scope not in existing_scopes:
    w.secrets.create_scope(scope=secret_scope)
    print(f"Created secret scope: {secret_scope}")
else:
    print(f"Secret scope already exists: {secret_scope}")

## Step 3: Garmin Authentication

**First run:** Uses your email/password to get OAuth tokens, then saves them to Databricks Secrets.

**Subsequent runs:** Loads saved tokens from Secrets. No email/password needed.

In [ ]:
import json
from pathlib import Path
from garminconnect import Garmin

tokenstore = Path.home() / ".garminconnect"
tokenstore.mkdir(parents=True, exist_ok=True)
token_path = tokenstore / "garmin_tokens.json"

has_saved_tokens = False
try:
    saved_tokens = dbutils.secrets.get(scope=secret_scope, key="garmin_oauth_tokens")
    if saved_tokens and saved_tokens.strip():
        token_path.write_text(saved_tokens)
        has_saved_tokens = True
        print("Loaded saved tokens from Databricks Secrets")
except Exception:
    print("No saved tokens found in Secrets")

client = None

if has_saved_tokens:
    try:
        client = Garmin()
        client.login(str(tokenstore))
        print("Garmin: authenticated using saved tokens")
    except Exception as e:
        print(f"Saved tokens expired or invalid: {e}")
        client = None

if client is None:
    if not garmin_email or not garmin_password:
        raise ValueError(
            "No valid saved tokens and no email/password provided. "
            "Set the garmin_email and garmin_password widgets for the first run."
        )
    print(f"Logging in with email: {garmin_email}")
    client = Garmin(email=garmin_email, password=garmin_password)
    client.login(str(tokenstore))
    print("Garmin: fresh login successful, tokens saved locally")

    fresh_tokens = token_path.read_text()
    w.secrets.put_secret(scope=secret_scope, key="garmin_oauth_tokens", string_value=fresh_tokens)
    print(f"Tokens saved to Databricks Secrets (scope={secret_scope}, key=garmin_oauth_tokens)")
    print("You can clear the email/password widgets now -- future runs use saved tokens.")

## Step 4: Determine Target Date

In [ ]:
from datetime import date, timedelta

if target_date_str and target_date_str.strip():
    target_date = date.fromisoformat(target_date_str.strip())
else:
    target_date = date.today() - timedelta(days=1)

print(f"Target date: {target_date}")

## Step 5: Extract Data from Garmin Connect API

Pulls all 10 API endpoints for the target date.

In [ ]:
ds = target_date.isoformat()

EXTRACTORS = [
    ("daily_stats", "get_stats"),
    ("heart_rates", "get_heart_rates"),
    ("sleep", "get_sleep_data"),
    ("stress", "get_stress_data"),
    ("hrv", "get_hrv_data"),
    ("spo2", "get_spo2_data"),
    ("body_battery", "get_body_battery"),
    ("steps", "get_steps_data"),
    ("respiration", "get_respiration_data"),
]

raw_by_type = {}

for record_type, method_name in EXTRACTORS:
    try:
        raw_by_type[record_type] = getattr(client, method_name)(ds)
        print(f"  {record_type}: OK")
    except Exception as e:
        print(f"  {record_type}: FAILED ({e})")

try:
    raw_by_type["activities"] = client.get_activities_fordate(ds)
    print(f"  activities: OK")
except Exception as e:
    print(f"  activities: FAILED ({e})")

print(f"\nExtracted {len(raw_by_type)} categories for {ds}")

## Step 6: Build Bronze Rows and Write to Delta

Each API category becomes one row in the VARIANT-based bronze table.

In [ ]:
import uuid
from datetime import datetime, timezone


def to_bronze_row(raw_data, record_type, dev_id, target_ds):
    now = datetime.now(timezone.utc).isoformat()
    body = {
        "source": "garmin_connect",
        "device_id": dev_id,
        "date": target_ds,
        "data": raw_data,
    }
    headers = {
        "Content-Type": "application/json",
        "X-Platform": "garmin_connect",
        "X-Record-Type": record_type,
        "X-Device-Id": dev_id,
        "X-Upload-Timestamp": now,
    }
    return {
        "record_id": str(uuid.uuid4()),
        "ingested_at": now,
        "body_json": json.dumps(body, default=str),
        "headers_json": json.dumps(headers),
        "record_type": record_type,
    }


bronze_rows = []
for record_type, raw_data in raw_by_type.items():
    if raw_data is None:
        continue
    bronze_rows.append(to_bronze_row(raw_data, record_type, device_id, ds))

print(f"Built {len(bronze_rows)} bronze rows:")
for row in bronze_rows:
    body_size = len(row['body_json'])
    print(f"  {row['record_type']:20s}  body={body_size:>6,} bytes")

In [ ]:
from pyspark.sql import Row
from pyspark.sql.types import StringType, StructField, StructType

staging_schema = StructType([
    StructField("record_id", StringType(), False),
    StructField("ingested_at", StringType(), False),
    StructField("body_json", StringType(), False),
    StructField("headers_json", StringType(), False),
    StructField("record_type", StringType(), True),
])

rows = [Row(**r) for r in bronze_rows]
staging_df = spark.createDataFrame(rows, schema=staging_schema)
staging_df.createOrReplaceTempView("garmin_staging")

spark.sql(f"""
    INSERT INTO {bronze_table} (record_id, ingested_at, body, headers, record_type)
    SELECT
        record_id,
        CAST(ingested_at AS TIMESTAMP),
        parse_json(body_json),
        parse_json(headers_json),
        record_type
    FROM garmin_staging
""")

print(f"Wrote {len(bronze_rows)} rows to {bronze_table}")

## Step 7: Save Refreshed Tokens

The garminconnect library may have refreshed the OAuth token during API calls.

In [ ]:
refreshed_tokens = token_path.read_text()
w.secrets.put_secret(scope=secret_scope, key="garmin_oauth_tokens", string_value=refreshed_tokens)
print(f"Refreshed tokens saved to Databricks Secrets")

## Step 8: Verify -- Your Garmin Data in Databricks

Query the bronze table using VARIANT path expressions.

In [ ]:
print(f"=== {bronze_table} ===\n")

total_df = spark.sql(f"SELECT COUNT(*) as total_rows FROM {bronze_table}")
display(total_df)

In [ ]:
print("--- Record types ---")
types_df = spark.sql(f"""
    SELECT
        record_type,
        COUNT(*) as cnt,
        MIN(ingested_at) as earliest,
        MAX(ingested_at) as latest
    FROM {bronze_table}
    GROUP BY record_type
    ORDER BY cnt DESC
""")
display(types_df)

In [ ]:
print("--- Daily Stats (your health metrics) ---")
stats_df = spark.sql(f"""
    SELECT
        body:date::STRING AS day,
        body:data:restingHeartRate::INT AS resting_hr,
        body:data:totalSteps::INT AS steps,
        body:data:totalKilocalories::INT AS total_calories,
        body:data:activeKilocalories::INT AS active_calories,
        body:data:floorsAscended::INT AS floors,
        body:data:averageStressLevel::INT AS avg_stress,
        body:data:vO2MaxValue::DOUBLE AS vo2_max
    FROM {bronze_table}
    WHERE record_type = 'daily_stats'
    ORDER BY day DESC
""")
display(stats_df)

In [ ]:
print("--- Sleep Data ---")
sleep_df = spark.sql(f"""
    SELECT
        body:date::STRING AS day,
        ROUND(body:data:dailySleepDTO:sleepTimeSeconds::INT / 3600.0, 1) AS sleep_hours,
        ROUND(body:data:dailySleepDTO:deepSleepSeconds::INT / 60.0, 0) AS deep_min,
        ROUND(body:data:dailySleepDTO:lightSleepSeconds::INT / 60.0, 0) AS light_min,
        ROUND(body:data:dailySleepDTO:remSleepSeconds::INT / 60.0, 0) AS rem_min,
        body:data:dailySleepDTO:sleepScores:overall:value::INT AS sleep_score
    FROM {bronze_table}
    WHERE record_type = 'sleep'
    ORDER BY day DESC
""")
display(sleep_df)

In [ ]:
print("--- HRV Data ---")
hrv_df = spark.sql(f"""
    SELECT
        body:date::STRING AS day,
        body:data:hrvSummary:weeklyAvg::DOUBLE AS hrv_weekly_avg_ms,
        body:data:hrvSummary:lastNight5MinHigh::DOUBLE AS hrv_last_night_ms,
        body:data:hrvSummary:status::STRING AS hrv_status
    FROM {bronze_table}
    WHERE record_type = 'hrv'
    ORDER BY day DESC
""")
display(hrv_df)

In [ ]:
print("--- All Sources in Bronze ---")
platform_df = spark.sql(f"""
    SELECT
        headers:"X-Platform"::STRING AS platform,
        record_type,
        COUNT(*) AS cnt
    FROM {bronze_table}
    GROUP BY 1, 2
    ORDER BY 1, 2
""")
display(platform_df)

print("\nDone! Your Garmin data is in Databricks.")